# Generate `mimic_cxr_cleaned.csv`

Chạy notebook này **một lần** để tạo file `mimic_cxr_cleaned.csv`, sau đó upload lên Kaggle Dataset `mimic-cxr-reported`.

**Datasets cần đính kèm:**
- **`mimic-cxr-jpg-lite`** — JPG images + metadata CSVs
- **`mimic-cxr-reported`** — radiology `.txt` reports

Sau khi chạy xong, file `mimic_cxr_cleaned.csv` sẽ xuất hiện trong Output. Upload nó vào dataset `mimic-cxr-reported` trên Kaggle.

In [ ]:
import os
import glob
import re
import yaml
import pandas as pd
from multiprocessing import Pool, cpu_count

# ── Load Kaggle dataset config ────────────────────────────────────────────────
with open("configs/kaggle_datasets.yaml") as f:
    CFG = yaml.safe_load(f)

IMAGES_SLUG   = CFG["datasets"]["images"]["slug"]
REPORTS_SLUG  = CFG["datasets"]["reports"]["slug"]
REQUIRED_CSVS = CFG["datasets"]["images"]["required_files"]
TXT_GLOB      = CFG["datasets"]["reports"]["txt_glob"]
REPORTS_LOCAL = CFG["working"]["cleaned_csv_path"]

def find_mount(slug):
    for root in CFG["mount_search_roots"]:
        candidate = os.path.join(root, slug)
        if os.path.isdir(candidate):
            return candidate
    matches = glob.glob(f"/kaggle/input/**/{slug}", recursive=True)
    return matches[0] if matches else None

# ── Images + metadata CSVs ───────────────────────────────────────────────────
KAGGLE_INPUT = find_mount(IMAGES_SLUG)
if not KAGGLE_INPUT:
    raise FileNotFoundError(
        f"Dataset '{IMAGES_SLUG}' not attached. Add it via Kaggle: Add Data → Datasets."
    )
IMAGE_ROOT = KAGGLE_INPUT
print(f"KAGGLE_INPUT (images + CSVs): {KAGGLE_INPUT}")

# ── Reports (.txt files) ─────────────────────────────────────────────────────
REPORTS_ROOT = find_mount(REPORTS_SLUG)
if not REPORTS_ROOT:
    raise FileNotFoundError(
        f"Dataset '{REPORTS_SLUG}' not attached. Add it via Kaggle: Add Data → Datasets."
    )
print(f"REPORTS_ROOT (txt reports):   {REPORTS_ROOT}")

# ── Verify metadata CSVs ─────────────────────────────────────────────────────
for fname in REQUIRED_CSVS:
    path = os.path.join(KAGGLE_INPUT, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing: {path}")
print("All metadata CSVs present.")

# ── Parse helpers ─────────────────────────────────────────────────────────────
SECTION_PAT = re.compile(
    r"(FINAL REPORT|EXAMINATION|INDICATION|TECHNIQUE|COMPARISON|HISTORY|"
    r"FINDINGS|IMPRESSION|RECOMMENDATION|NOTIFICATION|CLINICAL HISTORY|REASON FOR EXAMINATION|"
    r"WET READ|WET READ VERSION):",
    re.IGNORECASE,
)

def extract_findings(text):
    parts = SECTION_PAT.split(text)
    sections = {}
    for i in range(1, len(parts), 2):
        sections[parts[i].upper().strip()] = parts[i + 1].strip()
    return sections.get("FINDINGS") or sections.get("IMPRESSION") or ""

def parse_one_report(txt_path):
    try:
        study_id = int(os.path.basename(txt_path)[1:-4])
        with open(txt_path, encoding="utf-8") as f:
            findings = extract_findings(f.read()).replace("\n", " ").strip()
    except (ValueError, OSError):
        return None
    if not findings:
        return None
    return (study_id, findings, os.path.basename(txt_path))

# ── Step 1: Collect all .txt report files ────────────────────────────────────
txt_files = glob.glob(f"{REPORTS_ROOT}/{TXT_GLOB}", recursive=True)
print(f"Found {len(txt_files)} report .txt files")
if len(txt_files) == 0:
    raise FileNotFoundError(
        f"No .txt report files found under {REPORTS_ROOT}/{TXT_GLOB}. "
        "Check that the reports dataset has the correct directory structure."
    )

# ── Step 2: Parse reports in parallel ─────────────────────────────────────────
n_workers = max(1, cpu_count())
print(f"Parsing with {n_workers} CPU workers…")
with Pool(processes=n_workers) as pool:
    results = pool.map(parse_one_report, txt_files, chunksize=500)

rows = [r for r in results if r is not None]
reports = pd.DataFrame(rows, columns=["study_id", "findings", "Note_file"])
print(f"Extracted {len(reports)} reports with non-empty findings")

# ── Step 3: Merge with metadata ───────────────────────────────────────────────
metadata = pd.read_csv(f"{KAGGLE_INPUT}/mimic-cxr-2.0.0-metadata.csv")
df = metadata.merge(reports, on="study_id", how="inner")

# ── Step 4: Vectorized path construction ─────────────────────────────────────
subj = df["subject_id"].astype(str)
sid  = df["study_id"].astype(str)
df["Img_Folder"]   = "p" + subj.str[:2] + "/p" + subj + "/s" + sid
df["Img_Filename"] = df["dicom_id"].astype(str) + ".jpg"
df = df[["dicom_id", "findings", "Img_Folder", "Img_Filename", "Note_file"]]

# ── Step 5: Filter to images that exist on disk ───────────────────────────────
print("Scanning IMAGE_ROOT for existing JPGs…")
all_jpgs = set()
for root, _, files in os.walk(IMAGE_ROOT):
    rel = os.path.relpath(root, IMAGE_ROOT)
    for fname in files:
        if fname.endswith(".jpg"):
            all_jpgs.add(f"{rel}/{fname}" if rel != "." else fname)
print(f"Found {len(all_jpgs)} JPGs on disk")

rel_paths = df["Img_Folder"] + "/" + df["Img_Filename"]
df = df[rel_paths.isin(all_jpgs)].reset_index(drop=True)

df.to_csv(REPORTS_LOCAL, index=False)
print(f"\nSaved {REPORTS_LOCAL}: {len(df)} rows")
print(f"\nSample:")
print(df[["Img_Folder", "Img_Filename"]].head(3).to_string())
print(f"\n✅ Done! Download {os.path.basename(REPORTS_LOCAL)} from Output and upload to your '{REPORTS_SLUG}' Kaggle dataset.")